# Does Anything Move The Score By 0.1?

In [ ]:
import hashlib
import time

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from comfyui_image_scorer.application.analysis.score_influence import (
    ScoreInfluenceCache,
    ScoreInfluenceCacheParams,
    analyze_sparse_combination_influence,
    build_condition_data,
    load_score_influence_cache,
    read_condition_elements,
    save_score_influence_cache,
)
from comfyui_image_scorer.core.filesystem.paths import text_data_file
from comfyui_image_scorer.core.io.serialization import load_single_jsonl
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)
from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    replay_ratings,
    trueskill_score_from_rating,
)
from comfyui_image_scorer.domain.training.frequent_combinations import (
    create_image_property_matrix,
    generate_combinations_sparse,
)
from comfyui_image_scorer.domain.training.statistics import (
    MaxStatisticResult,
    max_statistic_bootstrap,
)
from comfyui_image_scorer.infrastructure.loading.training_loader import training_loader
from comfyui_image_scorer.infrastructure.ml_models.plot import PlotManager
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)


In [ ]:
SCORE_FORM: str = "public"  # Score scale: public or trueskill.
MOVE: float = 0.10  # Effect threshold in public-score points.
MIN_MATCHING_IMAGES: int = 100  # Minimum support; direct cache-folder key.
MAX_MATCHING_IMAGES: int = (
    25000  # Maximum support of a measured combination; conditions matching every analyzed image are dropped at level 1.
)
NUMERIC_BINS: int = 4  # Numeric quantile bins; changes matrix and cache folder.
MAX_DEPTH: int = 3  # Maximum conditions per combination.
MEMORY_BUDGET_GB: float = 8.0  # Memory target used to size search batches.
PROPERTY_BLOCK_SIZE: int = 4  # Properties per blocked multiplication tile.
FOLD_COUNT: int = 5  # Number of cross-validation folds.
PERMUTATION_ROUNDS: int = 20  # Shuffled-score null rounds.
BOOTSTRAP_ITERATIONS: int = 2000  # Resamples for the single-condition threshold.
SEED: int = 20251015  # Seed for reproducible folds and resampling.
TOP_CHART: int = 100  # Number of top single conditions to chart.
EXAMPLE_GRID: int = 8  # Number of image-grid columns.

# Available SCORE_FORM names and their score conversion functions.
SCORE_FORMS = {
    "public": public_score_from_rating,
    "trueskill": trueskill_score_from_rating,
}
if SCORE_FORM not in SCORE_FORMS:
    raise ValueError(f"SCORE_FORM must be one of {sorted(SCORE_FORMS)}")
score_of = SCORE_FORMS[SCORE_FORM]

comparison_rows = training_loader.load_comparison_rows()
replayed = replay_ratings(comparison_rows, order="default")
ratings = {name: r for name, (r, _) in replayed.items()}
spreads = {
    f: float(np.std([fn(r) for r in ratings.values()], ddof=1))
    for f, fn in SCORE_FORMS.items()
}
target_sd = spreads[SCORE_FORM]
THRESHOLD = (
    MOVE * target_sd / spreads["public"]
)  # Convert MOVE to the selected score scale.
scores = {name: score_of(r) for name, r in ratings.items()}
target_values = np.array(list(scores.values()), dtype=np.float64)

logger.info(
    "target '%s': min %.4f max %.4f mean %.4f sd %.4f",
    SCORE_FORM,
    target_values.min(),
    target_values.max(),
    target_values.mean(),
    target_sd,
)
logger.info(
    "move %.2f public = %.4f in '%s' units (%.0f%% of sd)",
    MOVE,
    THRESHOLD,
    SCORE_FORM,
    100.0 * THRESHOLD / target_sd,
)


In [ ]:
read_started = time.perf_counter()
elements_by_image = read_condition_elements(
    scores,
    load_single_jsonl(text_data_file),
    minimum_matching_images=MIN_MATCHING_IMAGES,
    numeric_bins=NUMERIC_BINS,
)
condition_data = build_condition_data(
    elements_by_image,
    scores,
    minimum_matching_images=MIN_MATCHING_IMAGES,
)
condition_names = condition_data.condition_names
image_names = condition_data.image_names
row_scores = condition_data.row_scores
property_image_bitmasks = condition_data.property_image_bitmasks

logger.info(
    "%d conditions survive level 1 (of %d raw) in %.0fs",
    len(condition_names),
    condition_data.raw_condition_count,
    time.perf_counter() - read_started,
)


In [ ]:
# Rows in each worker matrix are the union of that batch's matching image IDs.
# Generate supported property combinations and cache sparse matches by level.
apriori_started = time.perf_counter()
property_count: int = len(condition_names)
image_count: int = len(row_scores)


image_property_matrix = create_image_property_matrix(
    property_image_bitmasks, image_count
)
parent_combinations = generate_combinations_sparse(
    image_property_matrix=image_property_matrix,
    image_row_ids=np.arange(image_count, dtype=np.int64),
    max_combination_size=MAX_DEPTH,
    minimum_matching_images=MIN_MATCHING_IMAGES,
    maximum_matching_images=MAX_MATCHING_IMAGES,
    memory_budget_gb=MEMORY_BUDGET_GB,
    property_block_size=PROPERTY_BLOCK_SIZE,
)
all_combinations_count = len(parent_combinations) - 1

logger.info(
    "total combinations: %d in %.0fs",
    all_combinations_count,
    time.perf_counter() - apriori_started,
)


In [ ]:
# Twin conditions: labels that cover exactly the same images as a kept label.

label_rows: dict[tuple[str, str], list[int]] = {}
for row, name in enumerate(condition_data.image_names):
    for condition in elements_by_image[name]:
        label_rows.setdefault(condition, []).append(row)

equal_mask_groups: dict[bytes, list[tuple[str, str]]] = {}
for condition, rows in label_rows.items():
    image_mask = np.zeros(len(condition_data.image_names), dtype=np.uint8)
    image_mask[np.array(rows, dtype=np.int64)] = 1
    if MIN_MATCHING_IMAGES <= len(rows) < len(condition_data.image_names):
        equal_mask_groups.setdefault(np.packbits(image_mask).tobytes(), []).append(
            condition
        )

twin_of: dict[str, list[str]] = {}
kept_labels: list[str] = []
for group in equal_mask_groups.values():
    group.sort()
    kept = f"{group[0][0]}: {group[0][1]}"
    kept_labels.append(kept)
    twin_of.setdefault(kept, []).extend(f"{one[0]}: {one[1]}" for one in group[1:])

if set(kept_labels) != set(condition_names):
    raise RuntimeError("twin label rebuild disagrees with build_condition_data")
logger.info(
    "twin map: %d dropped labels across %d kept conditions",
    sum(len(twins) for twins in twin_of.values()),
    len(condition_names),
)


def combination_twin_labels(source_index: int) -> list[str]:
    """Labels of condition tuples whose image set equals one kept combination."""
    members = parent_combinations.combination_at(source_index)
    if len(members) == 1:
        return list(twin_of.get(condition_names[int(members[0])], []))
    levels = parent_combinations.levels
    winner = frozenset(int(member) for member in members)
    wanted_rows = np.sort(
        np.asarray(parent_combinations.image_rows_at(source_index), dtype=np.int64)
    )
    wanted_bytes = wanted_rows.tobytes()

    def supersets(level: int) -> list[int]:
        combinations = levels[level].combinations
        offsets = levels[level].offsets
        image_ids = levels[level].image_ids
        found = []
        for row in range(combinations.shape[0]):
            span = image_ids[offsets[row] : offsets[row + 1]]
            if span.size < wanted_rows.size:
                continue
            probes = np.searchsorted(span, wanted_rows)
            if (probes < span.size).all() and np.array_equal(span[probes], wanted_rows):
                found.append(row)
        return found

    def level_span(level: int, row: int) -> np.ndarray:
        return levels[level].image_ids[
            levels[level].offsets[row] : levels[level].offsets[row + 1]
        ]

    twins: set[str] = set()
    one_level_rows = supersets(0)
    if len(members) == 2:
        for left in range(len(one_level_rows)):
            for right in range(left + 1, len(one_level_rows)):
                a = int(levels[0].combinations[one_level_rows[left]][0])
                b = int(levels[0].combinations[one_level_rows[right]][0])
                if {a, b} == winner:
                    continue
                shared = np.intersect1d(
                    level_span(0, one_level_rows[left]),
                    level_span(0, one_level_rows[right]),
                    assume_unique=True,
                )
                if shared.tobytes() == wanted_bytes:
                    twins.add(" + ".join(condition_names[c] for c in sorted((a, b))))
        return sorted(twins)

    for pair_row in supersets(1):
        pair_members = tuple(int(value) for value in levels[1].combinations[pair_row])
        for column_row in one_level_rows:
            column = int(levels[0].combinations[column_row][0])
            if frozenset(pair_members) | {column} == winner:
                continue
            shared = np.intersect1d(
                level_span(1, pair_row), level_span(0, column_row), assume_unique=True
            )
            if shared.tobytes() == wanted_bytes:
                twins.add(
                    " + ".join(
                        condition_names[c] for c in sorted((*pair_members, column))
                    )
                )
    return sorted(twins)


In [ ]:
# Saved matching image rows refer to the original image axis.
cache_directory = getattr(parent_combinations, "cache_directory", None)
row_scores_sha256 = hashlib.sha256(
    np.ascontiguousarray(row_scores).tobytes()
).hexdigest()
analysis_cache_params = ScoreInfluenceCacheParams(
    row_scores_sha256=row_scores_sha256,
    minimum_matching_images=MIN_MATCHING_IMAGES,
    maximum_matching_images=MAX_MATCHING_IMAGES,
    fold_count=FOLD_COUNT,
    permutation_rounds=PERMUTATION_ROUNDS,
    seed=SEED,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    bootstrap_seed=SEED,
    score_form=SCORE_FORM,
    max_combination_size=MAX_DEPTH,
    numeric_bins=NUMERIC_BINS,
    memory_budget_gb=MEMORY_BUDGET_GB,
    property_block_size=PROPERTY_BLOCK_SIZE,
)
analysis_cache = load_score_influence_cache(cache_directory, analysis_cache_params)
family_cache_comparable = None
family = None
if analysis_cache is not None:
    combination_influence = analysis_cache.combination_influence
    family = MaxStatisticResult(
        observed=analysis_cache.family_observed,
        p_values=analysis_cache.family_p_values,
        critical_value=analysis_cache.family_critical_value,
        iterations=analysis_cache.family_iterations,
    )
    family_cache_comparable = analysis_cache.family_comparable
else:
    combination_influence = analyze_sparse_combination_influence(
        parent_combinations,
        row_scores,
        minimum_matching_images=MIN_MATCHING_IMAGES,
        maximum_matching_images=MAX_MATCHING_IMAGES,
        fold_count=FOLD_COUNT,
        permutation_rounds=PERMUTATION_ROUNDS,
        seed=SEED,
    )
fold_results = combination_influence.fold_results
lift_sum = combination_influence.lift_sum
lift_count = combination_influence.lift_count
matching_image_counts = combination_influence.matching_image_counts
null_lifts = combination_influence.null_lifts


In [ ]:
# Reuse the image-by-condition matrix built for combination enumeration.
# Its transpose has conditions in rows and images in columns.
condition_bits = image_property_matrix.T
condition_counts = condition_bits.sum(axis=1)
comparable = np.flatnonzero((condition_counts > 0) & (condition_counts < image_count))
if (
    "family" not in locals()
    or family is None
    or family_cache_comparable is None
    or not np.array_equal(family_cache_comparable, comparable)
):
    I_bool = condition_bits[comparable].T
    family = max_statistic_bootstrap(
        I_bool, row_scores, iterations=BOOTSTRAP_ITERATIONS, seed=SEED
    )
    if (
        "analysis_cache" in locals()
        and analysis_cache is None
        and "cache_directory" in locals()
        and cache_directory is not None
    ):
        save_score_influence_cache(
            cache_directory,
            ScoreInfluenceCache(
                combination_influence=combination_influence,
                family_comparable=comparable,
                family_observed=family.observed,
                family_p_values=family.p_values,
                family_critical_value=family.critical_value,
                family_iterations=family.iterations,
            ),
            analysis_cache_params,
            condition_names=condition_names,
        )
logger.info(
    "family-wise threshold: %.4f (%d of %d clear)",
    family.critical_value,
    int((family.p_values <= 0.05).sum()),
    len(comparable),
)


In [ ]:
# Chart
from matplotlib.lines import Line2D

single_df = pd.DataFrame(
    {
        "label": [condition_names[i] for i in comparable],
        "kind": "single",
        "influence": family.observed,
        "p_value": family.p_values,
        "n": condition_counts[comparable],
    }
).sort_values("influence", key=abs, ascending=False)

analyzed = np.flatnonzero(lift_count > 0)
combo_means = np.zeros(len(parent_combinations), dtype=np.float64)
combo_means[analyzed] = lift_sum[analyzed] / lift_count[analyzed]
combo_df = (
    pd.DataFrame(
        {
            "source_idx": [int(index) for index in analyzed],
            "label": [
                " + ".join(condition_names[p] for p in parent_combinations[i])
                for i in analyzed
            ],
            "kind": "combination",
            "influence": combo_means[analyzed],
            "p_value": np.nan,
            "n": matching_image_counts[analyzed],
            "depth": [len(parent_combinations[i]) for i in analyzed],
        }
    ).sort_values("influence", key=abs, ascending=False)
    if analyzed.size
    else pd.DataFrame()
)

chart = pd.concat(
    [
        single_df.head(TOP_CHART),
        combo_df.head(1) if not combo_df.empty else pd.DataFrame(),
    ],
    ignore_index=True,
).sort_values("influence", key=abs)
null_p95 = float(np.quantile(np.abs(null_lifts), 0.95))
observed_mean = float(combo_df.iloc[0]["influence"]) if not combo_df.empty else 0.0


def _row_twin_labels(row):
    if abs(row.influence) < THRESHOLD:
        return []
    if row.kind == "single":
        return list(twin_of.get(row.label, []))
    return combination_twin_labels(int(row.source_idx))


row_twin_lists = [_row_twin_labels(row) for row in chart.itertuples()]
row_twin_texts = ["\n".join(f"≈ {one}" for one in twins) for twins in row_twin_lists]


def _colour(row):
    if row.kind == "combination":
        return (
            "tab:green"
            if (abs(row.influence) >= THRESHOLD and abs(observed_mean) > null_p95)
            else "tab:orange" if abs(row.influence) >= THRESHOLD else "tab:grey"
        )
    return (
        "tab:green"
        if (abs(row.influence) >= THRESHOLD and row.p_value <= 0.05)
        else "tab:orange" if abs(row.influence) >= THRESHOLD else "tab:grey"
    )


row_gap = 0.6
tick_positions = []
tick_labels = []
cursor = 0.0
for row, twin_text in zip(chart.itertuples(), row_twin_texts):
    head = f"{row.label[:70]}  (n={row.n:,})"
    tick_labels.append(f"{head}\n{twin_text}" if twin_text else head)
    lines = twin_text.count("\n") + 1 if twin_text else 1
    tick_positions.append(cursor + lines / 2 - 0.5)
    cursor += lines + row_gap
total_height = cursor - row_gap
colours = [_colour(r) for r in chart.itertuples()]
figure, axis = plt.subplots(figsize=(12, min(36.0, max(6.0, 0.2 * total_height + 1.5))))
axis.barh(
    tick_positions,
    chart["influence"],
    color=colours,
    edgecolor="black",
    linewidth=0.4,
    alpha=0.85,
    hatch=["///" if r.kind == "combination" else "" for r in chart.itertuples()],
)
axis.axvline(THRESHOLD, color="tab:red", linestyle="--", linewidth=1.3)
axis.axvline(-THRESHOLD, color="tab:red", linestyle="--", linewidth=1.3)
axis.axvline(family.critical_value, color="grey", linestyle="-.", linewidth=1.3)
axis.axvline(-family.critical_value, color="grey", linestyle="-.", linewidth=1.3)
axis.axvline(null_p95, color="black", linestyle=":", linewidth=1.3)
axis.axvline(-null_p95, color="black", linestyle=":", linewidth=1.3)
axis.axvline(0.0, color="black", linewidth=1.0)
axis.set_yticks(tick_positions)
axis.set_ylim(-1.0, total_height + 1.0)
axis.set_yticklabels(tick_labels, fontsize=7.5, linespacing=1.35)
axis.set_xlabel(f"effect on '{SCORE_FORM}' target")
axis.set_title(f"What moves the {SCORE_FORM} score by {MOVE:.2f}?")
axis.grid(axis="x", linestyle=":", alpha=0.4)
legend_handles = [
    Line2D(
        [0],
        [0],
        color="tab:red",
        linestyle="--",
        linewidth=1.3,
        label=f"effect threshold (±{THRESHOLD:.2f})",
    ),
    Line2D(
        [0],
        [0],
        color="grey",
        linestyle="-.",
        linewidth=1.3,
        label=f"family noise floor (±{family.critical_value:.3f})",
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=":",
        linewidth=1.3,
        label=f"null permutation 95% (±{null_p95:.3f})",
    ),
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=1.0, label="zero (no effect)"
    ),
    Line2D(
        [0],
        [0],
        color="tab:green",
        marker="s",
        linestyle="None",
        markersize=6,
        label="passes effect & family thresholds",
    ),
    Line2D(
        [0],
        [0],
        color="tab:orange",
        marker="s",
        linestyle="None",
        markersize=6,
        label="|effect| ≥ move, not family-significant",
    ),
    Line2D(
        [0],
        [0],
        color="tab:grey",
        marker="s",
        linestyle="None",
        markersize=6,
        label="|effect| below move threshold",
    ),
    Line2D(
        [0],
        [0],
        color="tab:purple",
        marker="D",
        linestyle="None",
        markersize=5,
        label="≈ twin conditions covering exactly the same images",
    ),
]
axis.legend(handles=legend_handles, loc="lower right", fontsize=7, framealpha=0.9)
figure.tight_layout()
plt.show()

logger.info(
    "single conditions at or past %.2f: %d of %d",
    THRESHOLD,
    int(single_df["influence"].abs().ge(THRESHOLD).sum()),
    len(single_df),
)
logger.info(
    "of those, clearing family-wise threshold: %d",
    int(
        (
            (single_df["influence"].abs() >= THRESHOLD) & (single_df["p_value"] <= 0.05)
        ).sum()
    ),
)
if not combo_df.empty:
    logger.info(
        "best combination: %+.4f (%s)", observed_mean, combo_df.iloc[0]["label"]
    )
    logger.info(
        "permutation p-value: %.4f (floor %.4f)",
        (1 + int((null_lifts >= abs(observed_mean)).sum())) / (PERMUTATION_ROUNDS + 1),
        1.0 / (PERMUTATION_ROUNDS + 1),
    )
for row, twins in zip(chart.itertuples(), row_twin_lists):
    if twins:
        logger.info("twins of %s: %s", row.label, " | ".join(twins))


In [ ]:
def _resolve(filename):
    path = find_image_path(filename)
    return str(path) if path is not None else None


if combo_df is not None and not combo_df.empty:
    best_idx = int(analyzed[np.argmax(combo_means[analyzed])])
    best_parents = parent_combinations[best_idx]
    required = {condition_names[p] for p in best_parents}
    required_set = set(required)
    matching = sorted(
        name
        for name, elems in elements_by_image.items()
        if required_set.issubset({f"{f}: {v}" for f, v in elems})
    )
    picked = matching[:EXAMPLE_GRID] + matching[-EXAMPLE_GRID:]
    logger.info(
        "%d images carry all %d conditions | %s",
        len(matching),
        len(required),
        " + ".join(required),
    )
    PlotManager.plot_image_grid(
        filenames=picked,
        titles=[f"{scores[n]:.3f}" for n in picked],
        path_for=_resolve,
        columns=EXAMPLE_GRID,
        title=f"Images carrying every condition of:\n{' + '.join(required)}",
    )


### What Each Variable Does

| Variable | What it controls | Unit | What moving it does |
|---|---|---|---|
| `SCORE_FORM` | target score form | `"public"` / `"trueskill"` | changes numbers and threshold meaning |
| `MOVE` | effect worth confirming | score points | moves the red lines |
| `MIN_MATCHING_IMAGES` | minimum images matching a property combination | images | minimum count for every level |
| `NUMERIC_BINS` | quantile bins per numeric param | bins | coarser or finer condition labels |
| `MAX_DEPTH` | deepest combination searched | conditions | more interaction, rarer and noisier |
| `MEMORY_BUDGET_GB` | resident Boolean matrices, batch arrays, and matmul cast-buffer allowance | GB | full intersection output is excluded |
| `PROPERTY_BLOCK_SIZE` | property columns grouped per blocked matmul tile | properties | used by the alternate upper-triangular calculator |
| `FOLD_COUNT` | cross-validation folds | folds | more folds = tighter mean, smaller holdout |
| `PERMUTATION_ROUNDS` | shuffled reruns | rounds | null tail resolution (floor = 1/(rounds+1)) |
| `BOOTSTRAP_ITERATIONS` | resamples for family threshold | resamples | stability of noise floor |
| `SEED` | fixes split and shuffles | integer | reproducibility, not reliability |
| `TOP_CHART` | conditions on chart | rows | display only |
| `EXAMPLE_GRID` | image grid columns | images | display only |